<a href="https://colab.research.google.com/github/7625-k/Aircraft-Validation-Report/blob/main/aircraft.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import warnings

# Ignore all DeprecationWarning messages
warnings.filterwarnings("ignore", category=DeprecationWarning)


In [ ]:
import numpy as np
import pandas as pd

In [ ]:
np.random.seed(42)
n = 5000

rpm = np.random.randint(500, 3000, n)  # Engine rotational speed

noise_v = np.random.normal(0, 0.8, n)
vibration_rms = 1 + 0.002 * rpm + noise_v   # V=1+0.002*RPM+noise
vibration_rms = np.clip(vibration_rms, 0.5, 12)

noise_t = np.random.normal(0, 3, n)
temperature = 40 + 0.005 * rpm + 2 * vibration_rms + noise_t   # T = 40 + 0.005*RPM + 2V + noise
temperature = np.clip(temperature, 40, 120)

noise_a = np.random.normal(0, 4, n)
acoustic_db = 50 + 3 * vibration_rms + noise_a   # A = 50 + 3V + noise
acoustic_db = np.clip(acoustic_db, 50, 110)

risk = np.where(
    (vibration_rms > 6) |
    (temperature > 90) |
    (acoustic_db > 85),
    1, 0
)  # 0=Normal,1=High Risk


data = pd.DataFrame({
    "vibration_rms": vibration_rms,
    "temperature": temperature,
    "rpm": rpm,
    "acoustic_db": acoustic_db,
    "risk": risk
})
print("Aircraft Vibration Sensor Data :")
print(data.head())

Aircraft Vibration Sensor Data :
   vibration_rms  temperature   rpm  acoustic_db  risk
0       4.242117    56.649433  1360    63.911513     0
1       3.677850    56.398421  1794    60.597177     0
2       5.230087    55.321575  1630    67.736925     0
3       5.577866    53.460412  1595    70.819576     0
4       6.208026    66.616834  2138    70.650827     1


In [ ]:
data.to_csv("aircraft_sensor_data.csv", index=False)

Loading dataset

In [ ]:
df_pd = pd.read_csv("aircraft_sensor_data.csv")
print(df_pd.head())

   vibration_rms  temperature   rpm  acoustic_db  risk
0       4.242117    56.649433  1360    63.911513     0
1       3.677850    56.398421  1794    60.597177     0
2       5.230087    55.321575  1630    67.736925     0
3       5.577866    53.460412  1595    70.819576     0
4       6.208026    66.616834  2138    70.650827     1


EDA (Exploratory Data Analysis )

In [ ]:
print(df_pd.shape)

(5000, 5)


In [ ]:
print(df_pd.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   vibration_rms  5000 non-null   float64
 1   temperature    5000 non-null   float64
 2   rpm            5000 non-null   int64  
 3   acoustic_db    5000 non-null   float64
 4   risk           5000 non-null   int64  
dtypes: float64(3), int64(2)
memory usage: 195.4 KB
None


In [ ]:
print(df_pd.describe())


       vibration_rms  temperature          rpm  acoustic_db         risk
count    5000.000000  5000.000000  5000.000000  5000.000000  5000.000000
mean        4.488956    57.742774  1753.904400    63.581022     0.206000
std         1.666961     7.337813   718.028882     6.324236     0.404471
min         0.500000    40.000000   500.000000    50.000000     0.000000
25%         3.233410    52.131247  1143.750000    59.089124     0.000000
50%         4.481624    57.636295  1752.000000    63.507848     0.000000
75%         5.772187    63.437441  2381.000000    68.002169     0.000000
max         9.222111    78.136382  2997.000000    83.465113     1.000000


In [ ]:
print(df_pd["risk"].value_counts())


risk
0    3970
1    1030
Name: count, dtype: int64


Converting Pandas to Polars

In [ ]:
import polars as pl

df_polars = pl.from_pandas(data)

print(df_polars.head())


shape: (5, 5)
┌───────────────┬─────────────┬──────┬─────────────┬──────┐
│ vibration_rms ┆ temperature ┆ rpm  ┆ acoustic_db ┆ risk │
│ ---           ┆ ---         ┆ ---  ┆ ---         ┆ ---  │
│ f64           ┆ f64         ┆ i64  ┆ f64         ┆ i64  │
╞═══════════════╪═════════════╪══════╪═════════════╪══════╡
│ 4.242117      ┆ 56.649433   ┆ 1360 ┆ 63.911513   ┆ 0    │
│ 3.67785       ┆ 56.398421   ┆ 1794 ┆ 60.597177   ┆ 0    │
│ 5.230087      ┆ 55.321575   ┆ 1630 ┆ 67.736925   ┆ 0    │
│ 5.577866      ┆ 53.460412   ┆ 1595 ┆ 70.819576   ┆ 0    │
│ 6.208026      ┆ 66.616834   ┆ 2138 ┆ 70.650827   ┆ 1    │
└───────────────┴─────────────┴──────┴─────────────┴──────┘


Filtering

In [ ]:
high_risk = df_polars.filter(pl.col("risk") == 1)  # High risk machines
print(high_risk.head())


shape: (5, 5)
┌───────────────┬─────────────┬──────┬─────────────┬──────┐
│ vibration_rms ┆ temperature ┆ rpm  ┆ acoustic_db ┆ risk │
│ ---           ┆ ---         ┆ ---  ┆ ---         ┆ ---  │
│ f64           ┆ f64         ┆ i64  ┆ f64         ┆ i64  │
╞═══════════════╪═════════════╪══════╪═════════════╪══════╡
│ 6.208026      ┆ 66.616834   ┆ 2138 ┆ 70.650827   ┆ 1    │
│ 7.498119      ┆ 69.383168   ┆ 2635 ┆ 74.533861   ┆ 1    │
│ 6.885886      ┆ 64.578521   ┆ 2933 ┆ 72.928031   ┆ 1    │
│ 8.860784      ┆ 74.086155   ┆ 2824 ┆ 77.047158   ┆ 1    │
│ 7.100765      ┆ 65.21755    ┆ 2568 ┆ 75.323599   ┆ 1    │
└───────────────┴─────────────┴──────┴─────────────┴──────┘


In [ ]:
high_rpm = df_polars.filter(pl.col("rpm") > 2500)  # Machines with high RPM
print(high_rpm.head())

shape: (5, 5)
┌───────────────┬─────────────┬──────┬─────────────┬──────┐
│ vibration_rms ┆ temperature ┆ rpm  ┆ acoustic_db ┆ risk │
│ ---           ┆ ---         ┆ ---  ┆ ---         ┆ ---  │
│ f64           ┆ f64         ┆ i64  ┆ f64         ┆ i64  │
╞═══════════════╪═════════════╪══════╪═════════════╪══════╡
│ 5.341525      ┆ 62.387697   ┆ 2669 ┆ 63.108512   ┆ 0    │
│ 7.498119      ┆ 69.383168   ┆ 2635 ┆ 74.533861   ┆ 1    │
│ 5.743992      ┆ 65.887916   ┆ 2891 ┆ 60.81613    ┆ 0    │
│ 6.885886      ┆ 64.578521   ┆ 2933 ┆ 72.928031   ┆ 1    │
│ 8.860784      ┆ 74.086155   ┆ 2824 ┆ 77.047158   ┆ 1    │
└───────────────┴─────────────┴──────┴─────────────┴──────┘


In [ ]:
high_vrms = df_polars.filter(pl.col("vibration_rms") > 6) # High vibration machines
print(high_vrms.head())


shape: (5, 5)
┌───────────────┬─────────────┬──────┬─────────────┬──────┐
│ vibration_rms ┆ temperature ┆ rpm  ┆ acoustic_db ┆ risk │
│ ---           ┆ ---         ┆ ---  ┆ ---         ┆ ---  │
│ f64           ┆ f64         ┆ i64  ┆ f64         ┆ i64  │
╞═══════════════╪═════════════╪══════╪═════════════╪══════╡
│ 6.208026      ┆ 66.616834   ┆ 2138 ┆ 70.650827   ┆ 1    │
│ 7.498119      ┆ 69.383168   ┆ 2635 ┆ 74.533861   ┆ 1    │
│ 6.885886      ┆ 64.578521   ┆ 2933 ┆ 72.928031   ┆ 1    │
│ 8.860784      ┆ 74.086155   ┆ 2824 ┆ 77.047158   ┆ 1    │
│ 7.100765      ┆ 65.21755    ┆ 2568 ┆ 75.323599   ┆ 1    │
└───────────────┴─────────────┴──────┴─────────────┴──────┘


In [ ]:
high_acoustic_db = df_polars.filter(pl.col("acoustic_db") > 85) # High acoustic machines i.e is noisy machines
print(high_acoustic_db.head())

shape: (0, 5)
┌───────────────┬─────────────┬─────┬─────────────┬──────┐
│ vibration_rms ┆ temperature ┆ rpm ┆ acoustic_db ┆ risk │
│ ---           ┆ ---         ┆ --- ┆ ---         ┆ ---  │
│ f64           ┆ f64         ┆ i64 ┆ f64         ┆ i64  │
╞═══════════════╪═════════════╪═════╪═════════════╪══════╡
└───────────────┴─────────────┴─────┴─────────────┴──────┘


In [ ]:
high_rpmAndvibration = df_polars.filter(                #  High RPM + High Vibration
    (pl.col("rpm") > 2000) &
    (pl.col("vibration_rms") > 5)
)
print(high_rpmAndvibration.head())

shape: (5, 5)
┌───────────────┬─────────────┬──────┬─────────────┬──────┐
│ vibration_rms ┆ temperature ┆ rpm  ┆ acoustic_db ┆ risk │
│ ---           ┆ ---         ┆ ---  ┆ ---         ┆ ---  │
│ f64           ┆ f64         ┆ i64  ┆ f64         ┆ i64  │
╞═══════════════╪═════════════╪══════╪═════════════╪══════╡
│ 6.208026      ┆ 66.616834   ┆ 2138 ┆ 70.650827   ┆ 1    │
│ 5.341525      ┆ 62.387697   ┆ 2669 ┆ 63.108512   ┆ 0    │
│ 7.498119      ┆ 69.383168   ┆ 2635 ┆ 74.533861   ┆ 1    │
│ 5.067956      ┆ 55.806404   ┆ 2185 ┆ 61.598812   ┆ 0    │
│ 5.743992      ┆ 65.887916   ┆ 2891 ┆ 60.81613    ┆ 0    │
└───────────────┴─────────────┴──────┴─────────────┴──────┘


Aggregation

In [ ]:
agg_pl = df_polars.group_by("risk").mean()
print(agg_pl)


shape: (2, 5)
┌──────┬───────────────┬─────────────┬─────────────┬─────────────┐
│ risk ┆ vibration_rms ┆ temperature ┆ rpm         ┆ acoustic_db │
│ ---  ┆ ---           ┆ ---         ┆ ---         ┆ ---         │
│ i64  ┆ f64           ┆ f64         ┆ f64         ┆ f64         │
╞══════╪═══════════════╪═════════════╪═════════════╪═════════════╡
│ 1    ┆ 6.797133      ┆ 66.66238    ┆ 2612.36699  ┆ 70.498146   │
│ 0    ┆ 3.890109      ┆ 55.428619   ┆ 1531.179849 ┆ 61.786402   │
└──────┴───────────────┴─────────────┴─────────────┴─────────────┘


In [ ]:
df_polars.group_by("risk").count()


risk,count
i64,u32
0,3970
1,1030


In [ ]:
df_polars.select([
    pl.col("rpm").min().alias("min_rpm"),
    pl.col("rpm").max().alias("max_rpm"),
    pl.col("temperature").max().alias("max_temperature"),
    pl.col("vibration_rms").max().alias("max_vibration")
])


min_rpm,max_rpm,max_temperature,max_vibration
i64,i64,f64,f64
500,2997,78.136382,9.222111


# Data Validation (Great Expectations)

In [ ]:
!pip install great_expectations==0.17.23

In [ ]:
import great_expectations as gx


In [ ]:
!great_expectations init



  ___              _     ___                  _        _   _
 / __|_ _ ___ __ _| |_  | __|_ ___ __  ___ __| |_ __ _| |_(_)___ _ _  ___
| (_ | '_/ -_) _` |  _| | _|\ \ / '_ \/ -_) _|  _/ _` |  _| / _ \ ' \(_-<
 \___|_| \___\__,_|\__| |___/_\_\ .__/\___\__|\__\__,_|\__|_\___/_||_/__/
                                |_|
             ~ Always know what to expect from your data ~

This looks like an existing project that appears complete! You are ready to roll.



In [ ]:
context = gx.get_context()


  parsed_store_backend_id = store_backend_id_file_parser.parseString(



In [ ]:
suite = context.add_or_update_expectation_suite(
    expectation_suite_name="aircraft_suite"
)


In [ ]:
context.sources.add_or_update_pandas(
    name="pandas_datasource"
)


PandasDatasource(type='pandas', name='pandas_datasource', id=None, assets=[])

In [ ]:
datasource = context.get_datasource("pandas_datasource")


In [ ]:
data_asset = datasource.add_dataframe_asset(
    name="aircraft_data"
)


In [ ]:
batch_request = data_asset.build_batch_request(
    dataframe=df_pd
)


In [ ]:
validator = context.get_validator(
    batch_request=batch_request,
    expectation_suite_name="aircraft_suite"
)


  return datetime.utcnow().replace(tzinfo=utc)



In [ ]:
validator.expect_table_row_count_to_be_between(
    min_value=4000,
    max_value=6000
)


  warnings.warn(



Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "success": true,
  "expectation_config": {
    "expectation_type": "expect_table_row_count_to_be_between",
    "kwargs": {
      "min_value": 4000,
      "max_value": 6000,
      "batch_id": "pandas_datasource-aircraft_data"
    },
    "meta": {}
  },
  "result": {
    "observed_value": 5000
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [ ]:
validator.expect_column_values_to_not_be_null("rpm")


Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

  return datetime.utcnow().replace(tzinfo=utc)



{
  "success": true,
  "expectation_config": {
    "expectation_type": "expect_column_values_to_not_be_null",
    "kwargs": {
      "column": "rpm",
      "batch_id": "pandas_datasource-aircraft_data"
    },
    "meta": {}
  },
  "result": {
    "element_count": 5000,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": []
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [ ]:
validator.expect_column_values_to_be_between(
    "temperature",
    min_value=40,
    max_value=120
)


  warnings.warn(



Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

  return datetime.utcnow().replace(tzinfo=utc)



{
  "success": true,
  "expectation_config": {
    "expectation_type": "expect_column_values_to_be_between",
    "kwargs": {
      "min_value": 40,
      "max_value": 120,
      "column": "temperature",
      "batch_id": "pandas_datasource-aircraft_data"
    },
    "meta": {}
  },
  "result": {
    "element_count": 5000,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [ ]:
validator.expect_column_values_to_be_between(
    "rpm",
    min_value=500,
    max_value=3000
)


  warnings.warn(



Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

  return datetime.utcnow().replace(tzinfo=utc)



{
  "success": true,
  "expectation_config": {
    "expectation_type": "expect_column_values_to_be_between",
    "kwargs": {
      "min_value": 500,
      "max_value": 3000,
      "column": "rpm",
      "batch_id": "pandas_datasource-aircraft_data"
    },
    "meta": {}
  },
  "result": {
    "element_count": 5000,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [ ]:
validator.expect_column_values_to_be_in_set(
    "risk",
    value_set=[0, 1]
)


  warnings.warn(



Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

  return datetime.utcnow().replace(tzinfo=utc)



{
  "success": true,
  "expectation_config": {
    "expectation_type": "expect_column_values_to_be_in_set",
    "kwargs": {
      "value_set": [
        0,
        1
      ],
      "column": "risk",
      "batch_id": "pandas_datasource-aircraft_data"
    },
    "meta": {}
  },
  "result": {
    "element_count": 5000,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [ ]:
validator.save_expectation_suite()

In [ ]:
checkpoint = context.add_or_update_checkpoint(
    name="aircraft_checkpoint",
    validator=validator
)

In [ ]:
checkpoint_result = checkpoint.run()

  return datetime.utcnow().replace(tzinfo=utc)



Calculating Metrics:   0%|          | 0/27 [00:00<?, ?it/s]

  return datetime.utcnow().replace(tzinfo=utc)



In [ ]:
context.build_data_docs()

{'local_site': 'file:///content/gx/uncommitted/data_docs/local_site/index.html'}

In [ ]:
context.open_data_docs()

In [ ]:
docs_sites = context.get_docs_sites_urls()

print(docs_sites)

[{'site_name': 'local_site', 'site_url': 'file:///content/gx/uncommitted/data_docs/local_site/index.html'}]


In [ ]:
!zip -r aircraft_validation_report.zip gx/uncommitted/data_docs/


updating: gx/uncommitted/data_docs/ (stored 0%)
updating: gx/uncommitted/data_docs/local_site/ (stored 0%)
updating: gx/uncommitted/data_docs/local_site/index.html (deflated 79%)
updating: gx/uncommitted/data_docs/local_site/expectations/ (stored 0%)
updating: gx/uncommitted/data_docs/local_site/expectations/aircraft_suite.html (deflated 80%)
updating: gx/uncommitted/data_docs/local_site/validations/ (stored 0%)
updating: gx/uncommitted/data_docs/local_site/validations/aircraft_suite/ (stored 0%)
updating: gx/uncommitted/data_docs/local_site/validations/aircraft_suite/__none__/ (stored 0%)
updating: gx/uncommitted/data_docs/local_site/validations/aircraft_suite/__none__/20260218T150644.533916Z/ (stored 0%)
updating: gx/uncommitted/data_docs/local_site/validations/aircraft_suite/__none__/20260218T150644.533916Z/pandas_datasource-aircraft_data.html (deflated 82%)
updating: gx/uncommitted/data_docs/local_site/static/ (stored 0%)
updating: gx/uncommitted/data_docs/local_site/static/images/

In [ ]:
from google.colab import files
files.download("aircraft_validation_report.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Machine Learning

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import shap
import matplotlib.pyplot as plt
from fairlearn.metrics import MetricFrame, selection_rate, false_positive_rate, false_negative_rate
from IPython.display import IFrame

In [ ]:
df = pd.read_csv("aircraft_sensor_data.csv")  # Replace with your file path

#  Add synthetic sensitive attribute if missing

if "machine_type" not in df.columns:
    df["machine_type"] = np.random.choice(["A", "B"], size=df.shape[0])

print("Dataset preview:")
display(df.head())
print("\nColumns:", df.columns)


In [ ]:
features = ["rpm", "vibration_rms", "temperature", "acoustic_db", "machine_type"]
target = "risk"

X = df[features]
y = df[target]

X = pd.get_dummies(X, columns=["machine_type"], drop_first=True)  # creates machine_type_B

# Train-test split (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
# 8️⃣ Random Forest Training
# =========================
rf = RandomForestClassifier(n_estimators=150, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

In [ ]:
print("\n=== Random Forest Evaluation ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
# 🔟 Feature Importance (creative visualization)
# =========================
importances = rf.feature_importances_
feat_names = X_train.columns

plt.figure(figsize=(8,5))
bars = plt.barh(feat_names, importances, color=plt.cm.viridis(importances))
plt.title("🎯 Random Forest Feature Importance", fontsize=16)
plt.xlabel("Importance", fontsize=12)
plt.ylabel("Features", fontsize=12)

# Add value labels
for bar in bars:
    width = bar.get_width()
    plt.text(width + 0.005, bar.get_y() + bar.get_height()/2, f"{width:.2f}", va='center')
plt.show()

In [ ]:
# 1️⃣1️⃣ SHAP Explanations
# =========================
explainer = shap.TreeExplainer(rf)
shap_values = explainer(X_test)

# Summary plot (beeswarm)
shap.summary_plot(shap_values, X_test, plot_type="dot", show=True)

# Summary bar plot
shap.summary_plot(shap_values, X_test, plot_type="bar", show=True)

In [ ]:
# 1️⃣2️⃣ Bias / Fairness Analysis
# =========================
sensitive_feature = X_test["machine_type_B"]

metrics = {
    "accuracy": lambda y_true, y_pred: (y_true == y_pred).mean(),
    "selection_rate": selection_rate,
    "false_positive_rate": false_positive_rate,
    "false_negative_rate": false_negative_rate
}

mf = MetricFrame(metrics=metrics,
                 y_true=y_test,
                 y_pred=y_pred,
                 sensitive_features=sensitive_feature)

print("\n=== Bias / Fairness Analysis by Machine Type ===")
display(mf.by_group)
